# MXNet recommender on SageMaker — train and evaluate

Prepare MovieLens data, train the Gluon model, check HitRate@10, then decide between batch
precomputation and a real-time endpoint.

In [ ]:
%pip install -q mxnet==1.9.1 sagemaker pandas numpy

In [ ]:
import sagemaker, boto3, pandas as pd, numpy as np

session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = "REPLACE-WITH-DataBucket-OUTPUT"
prefix = "processed"
print(session.boto_region_name, bucket)

## 1. Prepare the interaction data

Leave-one-out by timestamp, 4 sampled negatives per positive. The negative sampling ratio is
the single knob that moves offline metrics most — try 1, 4 and 8.

In [ ]:
!python ../src/data/prepare.py --ratings ./ml-25m/ratings.csv --out ./data --implicit --negatives 4
!aws s3 sync ./data s3://{bucket}/{prefix}/

## 2. Train

In [ ]:
from sagemaker.mxnet import MXNet

estimator = MXNet(
    entry_point="train.py",
    source_dir="../src/training",
    role=role,
    framework_version="1.9.0",
    py_version="py38",
    instance_type="ml.g4dn.xlarge",
    instance_count=1,
    output_path=f"s3://{bucket}/models/",
    hyperparameters={"epochs": 8, "factors": 64, "batch-size": 4096, "implicit": 1},
)

estimator.fit({
    "train": f"s3://{bucket}/{prefix}/train/",
    "validation": f"s3://{bucket}/{prefix}/validation/",
})

## 3. Evaluate: HitRate@10 and NDCG@10

Offline AUC looks good long before recommendations are useful. For each held-out positive,
rank it against 99 sampled negatives and check whether it lands in the top 10.

In [ ]:
def hit_rate_at_k(scores_matrix, k=10):
    # scores_matrix[:, 0] is the held-out positive; the rest are sampled negatives
    ranks = (scores_matrix[:, 1:] >= scores_matrix[:, :1]).sum(axis=1)
    return float((ranks < k).mean())

def ndcg_at_k(scores_matrix, k=10):
    ranks = (scores_matrix[:, 1:] >= scores_matrix[:, :1]).sum(axis=1)
    gains = np.where(ranks < k, 1.0 / np.log2(ranks + 2), 0.0)
    return float(gains.mean())

## 4. Serve

Batch transform for the nightly top-N, endpoint only if you need cold-start scoring.

In [ ]:
transformer = estimator.transformer(
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/batch-output/",
    strategy="MultiRecord",
    assemble_with="Line",
)
transformer.transform(f"s3://{bucket}/{prefix}/score-input/", content_type="application/json", split_type="Line")
transformer.wait()

!python ../src/serving/load_recommendations.py --bucket {bucket} --table recommender-dev-recs